# SPEI Computation Pipeline — Vanuatu (ISIMIP-BCSD)

**Standardized Precipitation-Evapotranspiration Index**, timescales 1/3/6/12 months, for `rcp26` and `rcp85`.

### Methodology (Vicente-Serrano et al., 2010 — the reference SPEI paper; mirrors the R `SPEI` package / SPEIbase)

1. **PET** via **Hargreaves-Samani (1985)**, using `tasmax`, `tasmin`, `tas` and extraterrestrial radiation (Ra) from latitude + day-of-year (FAO-56, Allen et al., 1998, Eqs. 21–25, 52).
   - Chosen because only temperature (+ wind) fields are available — no surface radiation or humidity, which rules out full Penman-Monteith.
   - `sfcWind` is loaded but not yet consumed; a clearly marked extension point is included if you later want Penman-Monteith with assumed humidity.
2. **Monthly climatic water balance**: `D_t = P_t - PET_t`
3. **Accumulation** over k = 1, 3, 6, 12 months (rolling sum of D).
4. For **each grid cell × each calendar month** separately:
   - Fit a 3-parameter **log-logistic distribution** to `D_k` using unbiased **Probability Weighted Moments (PWM)**, calibrated **only** on `REFERENCE_PERIOD` (out-of-sample fitting — ETCCDI-style base-period discipline).
   - Apply the fitted CDF to the **full** time series and map through the inverse standard-normal (`scipy.stats.norm.ppf`) to get SPEI values.
5. Write one NetCDF per `(scenario, timescale)`.
6. **Validate**: reference-period SPEI should have mean ≈ 0, std ≈ 1 at every grid cell/month, by construction — the validation section checks this and flags cells that fail.

### Assumptions to verify for your dataset
- File naming: `INPUT_DIR/<scenario>/*<var>*.nc` (glob-based). Adjust `FILE_GLOB_TEMPLATE` if yours differs.
- `pr` units: kg m⁻² s⁻¹ (flux) or already mm — both auto-detected.
- `tas`/`tasmax`/`tasmin` units: Kelvin or Celsius — both auto-detected.
- Lat/lon coordinate names: auto-detected among `{lat, latitude}` / `{lon, longitude}`.
- Time coordinate: standard CF time, daily or monthly — auto-detected and resampled to monthly as needed.


## 1. Imports

In [ ]:
from __future__ import annotations

import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.special import gamma as gamma_fn
from scipy.stats import norm

warnings.filterwarnings("ignore", category=RuntimeWarning)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("SPEI")


## 2. Project configuration

In [ ]:
# --- Project paths -----------------------------------------------------
PROJECT_DIR = Path("/mnt/wcs-s3-d1/BOLT_ISIMIP_BCSD/VANUATU")
INPUT_DIR = PROJECT_DIR / "data/final"
OUTPUT_DIR = PROJECT_DIR / "data/interim/Indicators/SPEI"

BASE_DIR = INPUT_DIR          # contains rcp26/ and rcp85/
SCENARIOS = ["rcp26", "rcp85"]
VARIABLES = ["pr", "tas", "tasmax", "tasmin", "sfcWind"]

SPEI_TIMESCALES = [1, 3, 6, 12]
REFERENCE_PERIOD = ("1985-01-01", "2014-12-31")
ELEVATION_M = 0.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Glob pattern used to find files for a given scenario/variable.
# e.g. matches "pr_rcp26_bcsd_....nc", "tasmax_..._rcp85_....nc", etc.
FILE_GLOB_TEMPLATE = "*{var}*.nc"

# Chunking for dask (tune to your grid size / memory)
CHUNKS = {"time": -1, "lat": 60, "lon": 60}


## 3. Helpers: coordinate & unit detection

In [ ]:
def _lat_name(ds: xr.Dataset) -> str:
    for c in ("lat", "latitude", "Lat", "Latitude"):
        if c in ds.coords or c in ds.dims:
            return c
    raise KeyError(f"No recognizable latitude coordinate in {list(ds.coords)}")


def _lon_name(ds: xr.Dataset) -> str:
    for c in ("lon", "longitude", "Lon", "Longitude"):
        if c in ds.coords or c in ds.dims:
            return c
    raise KeyError(f"No recognizable longitude coordinate in {list(ds.coords)}")


def _is_kelvin(da: xr.DataArray) -> bool:
    """Heuristic: mean magnitude > 100 implies Kelvin."""
    m = float(da.mean(skipna=True).compute())
    return m > 100


def _is_flux_precip(da: xr.DataArray) -> bool:
    """Heuristic: typical daily precip flux (kg m-2 s-1) is ~1e-5; if the
    mean magnitude is tiny, it's a flux and needs *86400 conversion."""
    m = float(np.abs(da.mean(skipna=True)).compute())
    return m < 0.01


def _is_daily(ds: xr.Dataset) -> bool:
    dt = pd.to_datetime(ds["time"].values[:2])
    return (dt[1] - dt[0]).days <= 3


## 4. Loading

In [ ]:
def load_variable(scenario: str, var: str) -> xr.DataArray:
    """Load and concatenate all files for a variable/scenario into one
    lazily-chunked DataArray."""
    scen_dir = BASE_DIR / scenario
    pattern = FILE_GLOB_TEMPLATE.format(var=var)
    files = sorted(scen_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files found for {var} in {scen_dir} "
                                 f"(pattern={pattern})")
    log.info(f"[{scenario}/{var}] loading {len(files)} file(s)")
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=True,
        chunks=CHUNKS,
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )
    if var not in ds.data_vars:
        # fall back to first data var if the var name doesn't match exactly
        candidates = list(ds.data_vars)
        log.warning(f"'{var}' not found in dataset vars {candidates}; "
                     f"using '{candidates[0]}'")
        var = candidates[0]
    return ds[var]


## 5. Unit normalisation & monthly resampling

In [ ]:
def to_monthly_precip_mm(pr: xr.DataArray) -> xr.DataArray:
    """Return monthly total precipitation in mm."""
    daily = _is_daily(pr.to_dataset(name="pr"))
    if _is_flux_precip(pr):
        pr_mm_per_step = pr * 86400.0  # kg m-2 s-1 -> mm/day
    else:
        pr_mm_per_step = pr  # already mm

    if daily:
        monthly = pr_mm_per_step.resample(time="MS").sum(skipna=False)
    else:
        # already monthly-ish: if it's a flux, mean flux * seconds_in_month
        days_in_month = pr_mm_per_step["time"].dt.days_in_month
        if _is_flux_precip(pr):
            monthly = pr_mm_per_step * days_in_month
        else:
            monthly = pr_mm_per_step
    monthly.name = "pr"
    return monthly


def to_monthly_temp_C(da: xr.DataArray) -> xr.DataArray:
    """Return monthly mean temperature in Celsius."""
    daily = _is_daily(da.to_dataset(name="t"))
    t_c = (da - 273.15) if _is_kelvin(da) else da
    monthly = t_c.resample(time="MS").mean(skipna=False) if daily else t_c
    monthly.name = da.name
    return monthly


## 6. PET: Hargreaves-Samani

In [ ]:
def extraterrestrial_radiation_MJ(lat_deg: xr.DataArray, doy: xr.DataArray) -> xr.DataArray:
    """FAO-56 Eqs. 21-25: Ra in MJ m-2 day-1, for latitude (deg) and
    day-of-year. Broadcasts lat x time."""
    Gsc = 0.0820  # MJ m-2 min-1
    phi = np.deg2rad(lat_deg)
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365.0 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365.0 * doy - 1.39)
    # sunset hour angle, clipped for numerical safety at high latitudes
    x = -np.tan(phi) * np.tan(delta)
    x = x.clip(-1, 1)
    ws = np.arccos(x)
    Ra = (24 * 60 / np.pi) * Gsc * dr * (
        ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws)
    )
    return Ra  # MJ m-2 day-1


def compute_pet_hargreaves(tasmax_c, tasmin_c, tas_c, lat_name) -> xr.DataArray:
    """Monthly PET (mm/month) via Hargreaves-Samani (1985)."""
    lat = tasmax_c[lat_name]
    doy = tasmax_c["time"].dt.dayofyear  # mid-month day-of-year (MS timestamps)
    Ra = extraterrestrial_radiation_MJ(lat, doy)  # MJ m-2 day-1
    Ra_mm = Ra * 0.408  # -> mm/day equivalent

    tr = (tasmax_c - tasmin_c).clip(min=0)  # diurnal temp range, avoid neg
    pet_day = 0.0023 * Ra_mm * (tas_c + 17.8) * np.sqrt(tr)  # mm/day
    pet_day = pet_day.clip(min=0)

    days_in_month = pet_day["time"].dt.days_in_month
    pet_month = pet_day * days_in_month
    pet_month.name = "pet"
    return pet_month


## 7. Log-logistic fit via Probability Weighted Moments (PWM)\n\nExact formulation used by Vicente-Serrano et al. (2010) / the R `SPEI` package.

In [ ]:
def _fit_loglogistic_pwm(x: np.ndarray):
    """Fit 3-parameter log-logistic distribution to 1D array `x` using
    unbiased PWMs (Hosking, 1990). Returns (alpha, beta, gamma) or NaNs."""
    x = x[np.isfinite(x)]
    n = x.size
    if n < 8:
        return np.nan, np.nan, np.nan
    x_sorted = np.sort(x)
    i = np.arange(1, n + 1)
    # Unbiased PWM estimators (Hosking 1990, used by SPEI package)
    w0 = np.mean(x_sorted)
    w1 = np.sum((n - i) / (n * (n - 1)) * x_sorted) if n > 1 else np.nan
    w2 = np.sum(
        (n - i) * (n - i - 1) / (n * (n - 1) * (n - 2)) * x_sorted
    ) if n > 2 else np.nan

    beta = (2 * w1 - w0) / (6 * w1 - w0 - 6 * w2)
    if not np.isfinite(beta) or beta <= 0:
        return np.nan, np.nan, np.nan
    try:
        g1 = gamma_fn(1 + 1 / beta)
        g2 = gamma_fn(1 - 1 / beta)
    except Exception:
        return np.nan, np.nan, np.nan
    denom = g1 * g2
    if not np.isfinite(denom) or denom == 0:
        return np.nan, np.nan, np.nan
    alpha = (w0 - 2 * w1) * beta / denom
    gamma_ = w0 - alpha * denom
    if not np.isfinite(alpha) or alpha <= 0:
        return np.nan, np.nan, np.nan
    return alpha, beta, gamma_


def _loglogistic_cdf(x: np.ndarray, alpha, beta, gamma_):
    with np.errstate(invalid="ignore", divide="ignore"):
        z = (x - gamma_) / alpha
        z = np.where(z <= 0, np.nan, z)  # domain: x > gamma
        F = 1.0 / (1.0 + z ** (-beta))
    return F


def _fit_transform_1d(full_series: np.ndarray, ref_mask: np.ndarray) -> np.ndarray:
    """For one grid cell & one calendar month: fit on ref period, transform
    the full series to SPEI (standard normal quantiles)."""
    out = np.full(full_series.shape, np.nan, dtype="float64")
    ref_vals = full_series[ref_mask]
    if np.sum(np.isfinite(ref_vals)) < 8:
        return out
    alpha, beta, gamma_ = _fit_loglogistic_pwm(ref_vals)
    if not np.isfinite(alpha):
        return out
    F = _loglogistic_cdf(full_series, alpha, beta, gamma_)
    eps = 1e-6
    F = np.clip(F, eps, 1 - eps)
    spei = norm.ppf(F)
    valid = np.isfinite(full_series)
    out[valid] = spei[valid]
    return out


## 8. SPEI computation for one accumulated series

In [ ]:
def compute_spei(D_accum: xr.DataArray, ref_period: tuple[str, str]) -> xr.DataArray:
    """
    D_accum : DataArray (time, lat, lon) of k-month accumulated water balance.
    Returns : DataArray of SPEI, same shape, computed per grid-cell &
              per-calendar-month, fit strictly on `ref_period`.
    """
    times = pd.to_datetime(D_accum["time"].values)
    ref_mask_full = (times >= pd.Timestamp(ref_period[0])) & (
        times <= pd.Timestamp(ref_period[1])
    )
    ref_mask_da = xr.DataArray(ref_mask_full, dims="time", coords={"time": D_accum["time"]})

    def _month_block(da_month: xr.DataArray, mask_month: xr.DataArray) -> xr.DataArray:
        return xr.apply_ufunc(
            _fit_transform_1d,
            da_month,
            mask_month,
            input_core_dims=[["time"], ["time"]],
            output_core_dims=[["time"]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[np.float64],
        )

    ds = xr.Dataset({"D": D_accum, "ref_mask": ref_mask_da})
    grouped = ds.groupby("time.month")
    result = grouped.map(lambda g: _month_block(g["D"], g["ref_mask"]))
    result = result.sortby("time")
    result.name = "spei"
    result.attrs.update(
        long_name="Standardized Precipitation-Evapotranspiration Index",
        units="dimensionless",
        method="Vicente-Serrano et al. (2010); log-logistic (PWM fit); "
               "PET=Hargreaves-Samani (1985)",
        reference_period=f"{ref_period[0]} to {ref_period[1]}",
    )
    return result


## 9. Main pipeline

In [ ]:
def process_scenario(scenario: str) -> None:
    log.info(f"=== Scenario: {scenario} ===")

    pr_raw = load_variable(scenario, "pr")
    tas_raw = load_variable(scenario, "tas")
    tasmax_raw = load_variable(scenario, "tasmax")
    tasmin_raw = load_variable(scenario, "tasmin")
    # sfcWind loaded for completeness / future Penman-Monteith extension
    try:
        _ = load_variable(scenario, "sfcWind")
    except FileNotFoundError:
        log.warning(f"[{scenario}] sfcWind not found — not required for "
                     f"Hargreaves PET, continuing.")

    log.info(f"[{scenario}] normalising units & resampling to monthly")
    P = to_monthly_precip_mm(pr_raw)
    T = to_monthly_temp_C(tas_raw)
    Tmax = to_monthly_temp_C(tasmax_raw)
    Tmin = to_monthly_temp_C(tasmin_raw)

    # Align all variables on a common time/lat/lon grid
    P, T, Tmax, Tmin = xr.align(P, T, Tmax, Tmin, join="inner")

    lat_name = _lat_name(T.to_dataset(name="t"))

    log.info(f"[{scenario}] computing PET (Hargreaves-Samani)")
    PET = compute_pet_hargreaves(Tmax, Tmin, T, lat_name)
    PET = PET.transpose(*P.dims)

    log.info(f"[{scenario}] computing monthly water balance D = P - PET")
    D = (P - PET).astype("float64")
    D.name = "D"

    for k in SPEI_TIMESCALES:
        log.info(f"[{scenario}] SPEI-{k}: accumulating {k}-month water balance")
        D_k = D.rolling(time=k, min_periods=k).sum()

        log.info(f"[{scenario}] SPEI-{k}: fitting log-logistic "
                 f"(ref={REFERENCE_PERIOD}) & computing index")
        spei_k = compute_spei(D_k, REFERENCE_PERIOD)
        spei_k = spei_k.compute()  # trigger dask graph, materialize result

        out_path = OUTPUT_DIR / f"SPEI{k}_{scenario}.nc"
        spei_k.attrs["timescale_months"] = k
        spei_k.attrs["scenario"] = scenario
        encoding = {"spei": {"zlib": True, "complevel": 4, "dtype": "float32"}}
        spei_k.to_dataset(name="spei").to_netcdf(out_path, encoding=encoding)
        log.info(f"[{scenario}] SPEI-{k} written -> {out_path}")


## 10. Run the pipeline\n\nThis executes the full computation for every scenario and writes NetCDF outputs to `OUTPUT_DIR`. This is the expensive cell — expect it to take a while depending on grid size and number of years.

In [ ]:
for scenario in SCENARIOS:
    try:
        process_scenario(scenario)
    except FileNotFoundError as e:
        log.error(f"Skipping scenario '{scenario}': {e}")
log.info("All scenarios processed.")


## 11. Validation

By construction, SPEI fitted on the reference period should have, **within the reference period itself**, mean ≈ 0 and std ≈ 1 at every grid cell and calendar month (it's a standard-normal transform of the fitted CDF evaluated on the same data it was fit on). This is the standard sanity check used before trusting a SPEI product (and mirrors the "in-base consistency" checks used for ETCCDI-style standardized indices).

This section checks, for every `(scenario, timescale)`:
1. **Global reference-period mean/std** of SPEI (expect ≈ 0 / ≈ 1).
2. **Per-cell reference-period mean/std** — flags the fraction of grid cells outside a tolerance (indicates insufficient data / a bad fit at that pixel, e.g. too few valid months, degenerate distribution).
3. **NaN fraction** over land cells (excessive NaNs usually means an upstream data problem, not a SPEI bug).
4. A quick spatial map of ref-period mean/std for eyeballing.
5. A sample time series for a single grid cell (visual sanity check that SPEI oscillates around 0 and captures known wet/dry excursions).


In [ ]:
MEAN_TOL = 0.10     # acceptable |mean| deviation from 0 within ref period
STD_TOL = 0.15      # acceptable |std - 1| deviation within ref period

validation_summary = []

for scenario in SCENARIOS:
    for k in SPEI_TIMESCALES:
        path = OUTPUT_DIR / f"SPEI{k}_{scenario}.nc"
        if not path.exists():
            log.warning(f"Missing output, skipping validation: {path}")
            continue

        ds = xr.open_dataset(path)
        spei = ds["spei"]

        ref_start, ref_end = REFERENCE_PERIOD
        spei_ref = spei.sel(time=slice(ref_start, ref_end))

        # --- global stats ---
        global_mean = float(spei_ref.mean(skipna=True))
        global_std = float(spei_ref.std(skipna=True))

        # --- per-cell stats ---
        cell_mean = spei_ref.mean(dim="time", skipna=True)
        cell_std = spei_ref.std(dim="time", skipna=True)

        valid_cells = np.isfinite(cell_mean.values)
        n_valid = int(valid_cells.sum())

        mean_ok = np.abs(cell_mean.values[valid_cells]) <= MEAN_TOL
        std_ok = np.abs(cell_std.values[valid_cells] - 1) <= STD_TOL
        pct_mean_ok = 100 * mean_ok.mean() if n_valid else np.nan
        pct_std_ok = 100 * std_ok.mean() if n_valid else np.nan

        # --- NaN fraction (over the full series, not just ref period) ---
        nan_frac = float(np.isnan(spei.values).mean())

        validation_summary.append({
            "scenario": scenario,
            "timescale": k,
            "global_mean": round(global_mean, 4),
            "global_std": round(global_std, 4),
            "pct_cells_mean_ok": round(pct_mean_ok, 1) if n_valid else np.nan,
            "pct_cells_std_ok": round(pct_std_ok, 1) if n_valid else np.nan,
            "n_valid_cells": n_valid,
            "nan_frac_full_series": round(nan_frac, 4),
        })

        ds.close()

val_df = pd.DataFrame(validation_summary)
val_df


### Pass/fail flags\n\nRule of thumb: `global_mean` within ±0.05 of 0, `global_std` within ±0.10 of 1, and >95% of cells passing tolerance is a healthy result. Rows failing this should be investigated (usually: too few valid reference-period months for that region, or a units/upstream data issue).

In [ ]:
def _flag(row):
    ok = (
        abs(row["global_mean"]) <= 0.05
        and abs(row["global_std"] - 1) <= 0.10
        and row["pct_cells_mean_ok"] >= 95
        and row["pct_cells_std_ok"] >= 95
    )
    return "PASS" if ok else "CHECK"

val_df["flag"] = val_df.apply(_flag, axis=1)
val_df


### Spatial maps: reference-period mean & std (per scenario/timescale)

In [ ]:
for scenario in SCENARIOS:
    for k in SPEI_TIMESCALES:
        path = OUTPUT_DIR / f"SPEI{k}_{scenario}.nc"
        if not path.exists():
            continue
        ds = xr.open_dataset(path)
        spei_ref = ds["spei"].sel(time=slice(*REFERENCE_PERIOD))
        cell_mean = spei_ref.mean(dim="time", skipna=True)
        cell_std = spei_ref.std(dim="time", skipna=True)

        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        cell_mean.plot(ax=axes[0], cmap="RdBu_r", vmin=-0.3, vmax=0.3)
        axes[0].set_title(f"{scenario} SPEI-{k}: ref-period mean (expect ~0)")
        cell_std.plot(ax=axes[1], cmap="viridis", vmin=0.7, vmax=1.3)
        axes[1].set_title(f"{scenario} SPEI-{k}: ref-period std (expect ~1)")
        plt.tight_layout()
        plt.show()
        ds.close()


### Sample grid-cell time series\n\nPicks the spatial center of the domain as a representative cell and plots SPEI over the full period, for a visual sanity check (index should oscillate around 0, with excursions during known wet/dry spells).

In [ ]:
for scenario in SCENARIOS:
    fig, axes = plt.subplots(len(SPEI_TIMESCALES), 1, figsize=(11, 2.2 * len(SPEI_TIMESCALES)), sharex=True)
    for ax, k in zip(np.atleast_1d(axes), SPEI_TIMESCALES):
        path = OUTPUT_DIR / f"SPEI{k}_{scenario}.nc"
        if not path.exists():
            continue
        ds = xr.open_dataset(path)
        lat_name = _lat_name(ds)
        lon_name = _lon_name(ds)
        mid_lat = ds[lat_name].values[len(ds[lat_name]) // 2]
        mid_lon = ds[lon_name].values[len(ds[lon_name]) // 2]
        series = ds["spei"].sel({lat_name: mid_lat, lon_name: mid_lon}, method="nearest")

        ax.axhline(0, color="grey", lw=0.8)
        ax.plot(series["time"].values, series.values, lw=1)
        ax.axvspan(pd.Timestamp(REFERENCE_PERIOD[0]), pd.Timestamp(REFERENCE_PERIOD[1]),
                   color="orange", alpha=0.1, label="reference period")
        ax.set_title(f"{scenario} — SPEI-{k} @ ({mid_lat:.2f}, {mid_lon:.2f})")
        ax.set_ylabel("SPEI")
        ds.close()
    axes[0].legend(loc="upper right", fontsize=8) if len(SPEI_TIMESCALES) > 1 else None
    plt.tight_layout()
    plt.show()
